Ten skrypt buduje agenta AI przeznaczonego do przeprowadzania badań w internecie - Deep Research w wersji DIY

# Setup

In [ ]:
!uv pip install -qU pydantic-ai litellm tavily-python nest_asyncio


*   `pydantic-ai`: Biblioteka do walidacji danych i serializacji, rozszerzona o funkcje związane ze sztuczną inteligencją.
*   `litellm`:  Biblioteka ułatwiająca korzystanie z różnych modeli językowych (LLM) poprzez standaryzowany interfejs.
*   `tavily-python`: Biblioteka do korzystania z wyszukiwarki Tavily, która koncentruje się na dostarczaniu wyników bez reklam i śledzenia.
*   `nest_asyncio`:  Biblioteka pozwalająca na uruchamianie zagnieżdżonych pętli zdarzeń asynchronicznych w Pythonie, co jest przydatne w niektórych specyficznych przypadkach użycia.

In [ ]:
from pydantic_ai import Agent, RunContext, Tool
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.models.gemini import GeminiModel
from IPython.display import display, Markdown
from pydantic import BaseModel
from dataclasses import dataclass, field
from typing import List
from litellm import completion
from tavily import TavilyClient
import json
import litellm
import os
import nest_asyncio
from google.colab import userdata

nest_asyncio.apply()

Ten kod importuje różne biblioteki i klasy potrzebne do stworzenia agenta AI wykorzystującego modele językowe, wyszukiwarkę internetową oraz narzędzia do zarządzania danymi.

*   `from pydantic_ai import Agent, RunContext, Tool`: Importuje podstawowe komponenty z biblioteki `pydantic-ai`, które są kluczowe dla budowy agenta AI:
    *   `Agent`: Klasa reprezentująca agenta AI, który może wykonywać zadania.
    *   `RunContext`:  Klasa przechowująca informacje o kontekście wykonania agenta (np. historia interakcji).
    *   `Tool`: Klasa definiująca narzędzie, które agent może używać do wykonywania określonych zadań.

*   `from pydantic_ai.models.openai import OpenAIModel`: Importuje klasę `OpenAIModel`, która umożliwia korzystanie z modeli językowych OpenAI (np. GPT-3, GPT-4).
*   `from pydantic_ai.models.gemini import GeminiModel`: Importuje klasę `GeminiModel`, która umożliwia korzystanie z modeli językowych Google Gemini.
*   `from IPython.display import display, Markdown`:  Importuje funkcje do wyświetlania danych w środowisku interaktywnym (np. Jupyter Notebook). `display` służy do wyświetlania różnych typów obiektów, a `Markdown` pozwala na renderowanie tekstu sformatowanego jako Markdown.
*   `from pydantic import BaseModel`: Importuje klasę `BaseModel` z biblioteki `pydantic`, która służy do definiowania modeli danych z walidacją.
*   `from dataclasses import dataclass, field`: Importuje funkcje `dataclass` i `field` z modułu `dataclasses`, które ułatwiają tworzenie klas reprezentujących dane.
*   `from typing import List`: Importuje typ `List` do określania list w definicjach funkcji i klas.
*   `from litellm import completion`: Importuje funkcję `completion` z biblioteki `litellm`, która służy do wysyłania zapytań do modeli językowych.
*   `from tavily import TavilyClient`: Importuje klasę `TavilyClient` z biblioteki `tavily`, która umożliwia korzystanie z wyszukiwarki internetowej Tavily.
*   `import json`: Importuje moduł `json` do pracy z danymi w formacie JSON.
*   `import litellm`: Importuje całą bibliotekę `litellm`.
*   `import os`: Importuje moduł `os`, który zapewnia dostęp do funkcji systemu operacyjnego (np. odczytywanie zmiennych środowiskowych).
*   `import nest_asyncio`: Importuje bibliotekę `nest_asyncio`, która pozwala na uruchamianie zagnieżdżonych pętli zdarzeń asynchronicznych.
*   `from google.colab import userdata`: Importuje moduł `userdata` z pakietu `google.colab`, który umożliwia dostęp do danych użytkownika przechowywanych w Google Colab (np. kluczy API).

*   `nest_asyncio.apply()`: Ta linia kodu uruchamia bibliotekę `nest_asyncio`, co pozwala na poprawne działanie zagnieżdżonych pętli zdarzeń asynchronicznych, które mogą być wymagane przez niektóre funkcje w kodzie.

In [ ]:
class CFG:
    model = "gpt-4o-mini"
    MAX_WEB_SEARCH_LOOPS = 2


os.environ["TAVILY_API_KEY"] = userdata.get("tavily")
os.environ["OPENAI_API_KEY"] = userdata.get("openaivision")

tavily_client = TavilyClient()

litellm.set_verbose = False

# model = GeminiModel('gemini-1.5-flash-8b')
model = OpenAIModel(CFG.model)


Ten kod definiuje konfigurację, ustawia klucze API i inicjalizuje obiekty potrzebne do działania agenta AI.

*   `class CFG:`: Definiuje klasę o nazwie `CFG`, która służy jako kontener dla stałych konfiguracyjnych.
    *   `model = 'gpt-4o-mini'`: Ustawia domyślny model językowy na `'gpt-4o-mini'`.  Ten model będzie używany przez agenta AI do generowania odpowiedzi i wykonywania zadań.
    *   `MAX_WEB_SEARCH_LOOPS = 2`: Określa maksymalną liczbę iteracji, które agent może wykonać podczas wyszukiwania informacji w internecie. Ograniczenie to zapobiega niekończącym się pętlom wyszukiwania.

*   `os.environ['TAVILY_API_KEY'] = userdata.get('tavily')`: Pobiera klucz API dla wyszukiwarki Tavily z danych użytkownika w Google Colab (przechowywanych pod nazwą `'tavily'`) i ustawia go jako zmienną środowiskową o nazwie `TAVILY_API_KEY`.  Zmienne środowiskowe są używane do przechowywania poufnych informacji, takich jak klucze API.
*   `os.environ['OPENAI_API_KEY'] = userdata.get('openaivision')`: Pobiera klucz API dla OpenAI z danych użytkownika w Google Colab (przechowywanych pod nazwą `'openaivision'`) i ustawia go jako zmienną środowiskową o nazwie `OPENAI_API_KEY`.

*   `tavily_client = TavilyClient()`: Tworzy instancję klasy `TavilyClient`, która będzie używana do wysyłania zapytań do wyszukiwarki Tavily.
*   `litellm.set_verbose=False`: Wyłącza tryb verbose (szczegółowy) dla biblioteki `litellm`.  Oznacza to, że `litellm` nie będzie wyświetlać szczegółowych komunikatów podczas wykonywania zapytań do modeli językowych.
*   `# model = GeminiModel('gemini-1.5-flash-8b')`: Ta linia jest zakomentowana i pokazuje alternatywną konfigurację, w której agent używałby modelu Google Gemini o nazwie `'gemini-1.5-flash-8b'`.
*   `model = OpenAIModel(CFG.model)`: Tworzy instancję klasy `OpenAIModel`, która będzie używana do interakcji z modelem językowym OpenAI określonym w konfiguracji (`CFG.model`), czyli `'gpt-4o-mini'`.  Ten obiekt `model` jest kluczowy dla generowania tekstu i wykonywania innych zadań przez agenta AI.

# Funkcje

In [ ]:
query_writer_system_prompt = """Your goal is to generate targeted web search query.

The query will gather information related to a specific topic.

Topic:
{research_topic}

Return your query as a JSON object:
{{
    "query": "string",
    "aspect": "string",
    "rationale": "string"
}}
"""

In [ ]:
summarizer_system_prompt = """Your goal is to generate a high-quality summary of the web search results.

When EXTENDING an existing summary:
1. Seamlessly integrate new information without repeating what's already covered
2. Maintain consistency with the existing content's style and depth
3. Only add new, non-redundant information
4. Ensure smooth transitions between existing and new content

When creating a NEW summary:
1. Highlight the most relevant information from each source
2. Provide a concise overview of the key points related to the report topic
3. Emphasize significant findings or insights
4. Ensure a coherent flow of information

In both cases:
- Focus on factual, objective information
- Maintain a consistent technical depth
- Avoid redundancy and repetition
- DO NOT use phrases like "based on the new results" or "according to additional sources"
- DO NOT add a preamble like "Here is an extended summary ..." Just directly output the summary.
- DO NOT add a References or Works Cited section.
"""

In [ ]:
reflection_system_prompt = """You are an expert research assistant analyzing a summary about {research_topic}.

Your tasks:
1. Identify knowledge gaps or areas that need deeper exploration
2. Generate a follow-up question that would help expand your understanding
3. Focus on technical details, implementation specifics, or emerging trends that weren't fully covered

Ensure the follow-up question is self-contained and includes necessary context for web search.

Return your analysis as a JSON object:
{{
    "knowledge_gap": "string",
    "follow_up_query": "string"
}}"""

In [ ]:
def format_sources(sources):
    formatted_text = "Sources:\n\n"
    for i, source in enumerate(sources, start=1):
        formatted_text += (
            f"Source {i}:\n"
            f"Title: {source['title']}\n"
            f"Url: {source['url']}\n"
            f"Content: {source['content']}\n\n"
        )
    return formatted_text.strip()

Używa f-stringów (formatted string literals) do wstawiania wartości z słownika `source` i indeksu `i` do tekstu.
*   `f"Source {i}:\n"`: Dodaje numer źródła oraz znak nowej linii.
*   `f"Title: {source['title']}\n"`: Dodaje tytuł źródła oraz znak nowej linii.
*   `f"Url: {source['url']}\n"`: Dodaje adres URL źródła oraz znak nowej linii.
*   `f"Content: {source['content']}\n\n"`: Dodaje treść źródła oraz dwa znaki nowej linii, aby oddzielić kolejne źródła.

In [ ]:
@dataclass
class ResearchDeps:
    research_topic: str = None
    search_query: str = None
    current_summary: str = None
    final_summary: str = None
    sources: List[str] = field(default_factory=list)
    latest_web_search_result: str = None
    research_loop_count: int = 0

Wewnątrz klasy zdefiniowano następujące pola (atrybuty):

*   `research_topic: str = None`: Przechowuje temat badań jako ciąg znaków. Domyślna wartość to `None`, co oznacza, że początkowo temat nie jest znany.
*   `search_query: str = None`: Przechowuje aktualne zapytanie wyszukiwania jako ciąg znaków. Domyślna wartość to `None`.
*   `current_summary: str = None`: Przechowuje bieżące podsumowanie wyników badań jako ciąg znaków. Domyślna wartość to `None`.
*   `final_summary: str = None`: Przechowuje ostateczne podsumowanie wyników badań jako ciąg znaków. Domyślna wartość to `None`.
*   `sources: List[str] = field(default_factory=list)`: Przechowuje listę źródeł informacji (np. adresy URL, tytuły artykułów) jako listę ciągów znaków.  `field(default_factory=list)` zapewnia, że każda instancja klasy `ResearchDeps` ma własną, niezależną pustą listę dla tego pola. Jest to ważne, aby uniknąć problemów z współdzieleniem stanu między różnymi instancjami.
*   `latest_web_search_result: str = None`: Przechowuje wynik ostatniego wyszukiwania w internecie jako ciąg znaków. Domyślna wartość to `None`.
*   `research_loop_count: int = 0`: Przechowuje liczbę iteracji (pętli) wykonanych podczas procesu badawczego jako liczbę całkowitą. Domyślna wartość to 0.

In [ ]:
async def generate_search_query(ctx: RunContext[ResearchDeps]) -> str:
    """Generate a query for web search"""
    # logger.info("==== CALLING generate_search_query... ====")
    print("==== CALLING generate_search_query... ====")
    response = completion(
        model=CFG.model,
        messages=[
            {
                "content": query_writer_system_prompt.format(
                    research_topic=ctx.deps.research_topic
                ),
                "role": "system",
            },
            {"content": "Generate a query for Web search.", "role": "user"},
        ],
        max_tokens=500,
        format="json",
    )
    search_query = json.loads(response.choices[0].message.content)
    # print(f"====>search_query:{search_query}")
    ctx.deps.search_query = search_query["query"]
    return "perform_web_search"

Ta funkcja asynchroniczna generuje zapytanie do wyszukiwarki internetowej na podstawie tematu badań i aktualnego stanu procesu badawczego.

In [ ]:
async def perform_web_search(ctx: RunContext[ResearchDeps]) -> str:
    """Do search and collect information"""
    print("==== CALLING perform_web_search... ====")
    search_results = tavily_client.search(
        ctx.deps.search_query, include_raw_content=False, max_results=1
    )
    search_string = format_sources(search_results["results"])
    ctx.deps.sources.extend(search_results["results"])
    ctx.deps.latest_web_search_result = search_string
    ctx.deps.research_loop_count += 1
    return "summarize_sources"

Ta funkcja asynchroniczna wykonuje wyszukiwanie w internecie przy użyciu wygenerowanego zapytania, formatuje wyniki i aktualizuje stan procesu badawczego.

In [ ]:
async def summarize_sources(ctx: RunContext[ResearchDeps]) -> str:
    """Summarize the gathered sources"""
    print("==== CALLING summarize_sources... ====")
    current_summary = ctx.deps.current_summary
    most_recent_web_research = ctx.deps.latest_web_search_result
    if current_summary:
        user_prompt = (
            f"Extend the existing summary: {current_summary}\n\n"
            f"Include new search results: {most_recent_web_research} "
            f"That addresses the following topic: {ctx.deps.research_topic}"
        )

    else:
        user_prompt = (
            f"Generate a summary of these search results: {most_recent_web_research} "
            f"That addresses the following topic: {ctx.deps.research_topic}"
        )

    response = completion(
        model=CFG.model,
        messages=[
            {
                "content": summarizer_system_prompt.format(
                    research_topic=ctx.deps.research_topic
                ),
                "role": "system",
            },
            {"content": user_prompt, "role": "user"},
        ],
        max_tokens=1000,
    )
    ctx.deps.current_summary = response.choices[0].message.content
    return "reflect_on_summary"

Ta funkcja asynchroniczna podsumowuje zebrane źródła informacji i aktualizuje stan procesu badawczego.

*   `async def summarize_sources(ctx: RunContext[ResearchDeps]) -> str:`: Definiuje funkcję asynchroniczną o nazwie `summarize_sources`, która przyjmuje jeden argument:
    *   `ctx`: Obiekt typu `RunContext[ResearchDeps]`. Ten obiekt zawiera kontekst wykonania agenta, w tym dostęp do stanu procesu badawczego przechowywanego w obiekcie `ctx.deps`. Funkcja zwraca ciąg znaków (`str`).

*   `print("==== CALLING summarize_sources... ====")`: Wyświetla komunikat informujący o wywołaniu funkcji.
*   `current_summary = ctx.deps.current_summary`: Pobiera bieżące podsumowanie z obiektu `ResearchDeps`.
*   `most_recent_web_research = ctx.deps.latest_web_search_result`: Pobiera wynik ostatniego wyszukiwania w internecie z obiektu `ResearchDeps`.

*   `if current_summary:`: Sprawdza, czy istnieje już bieżące podsumowanie.
    *   Jeśli tak (podsumowanie istnieje): Tworzy zapytanie (`user_prompt`) dla modelu językowego, które prosi o rozszerzenie istniejącego podsumowania o nowe wyniki wyszukiwania.  Zapytanie zawiera:
        *   Istniejące podsumowanie (`current_summary`).
        *   Najnowsze wyniki wyszukiwania (`most_recent_web_research`).
        *   Temat badań (`ctx.deps.research_topic`).
    *   `else:` (podsumowania nie ma): Tworzy zapytanie, które prosi o wygenerowanie podsumowania na podstawie najnowszych wyników wyszukiwania i tematu badań.

*   `response = completion( ... )`: Wysyła zapytanie do modelu językowego za pomocą funkcji `completion`.
    *   `model = CFG.model`: Określa model językowy, który ma być użyty.
    *   `messages=[ ... ]`: Przekazuje listę wiadomości do modelu językowego:
        *   Wiadomość systemowa (`summarizer_system_prompt`), która ustawia kontekst dla modelu językowego i zawiera temat badań.
        *   Wiadomość użytkownika (`user_prompt`), która zawiera zapytanie o podsumowanie.
    *   `max_tokens=1000`: Ogranicza maksymalną długość odpowiedzi do 1000 tokenów.

*   `ctx.deps.current_summary = response.choices[0].message.content`: Zapisuje wygenerowane podsumowanie (z odpowiedzi modelu językowego) w polu `current_summary` obiektu `ResearchDeps`.
*   `return "reflect_on_summary"`: Zwraca ciąg znaków `"reflect_on_summary"`. Ten ciąg służy jako sygnał dla agenta AI, aby wykonał następny krok, czyli przemyślenie (refleksja) nad wygenerowanym podsumowaniem.

In [ ]:
async def reflect_on_summary(ctx: RunContext[ResearchDeps]) -> str:
    """Reflect on the summary and generate a follow-up query"""
    # logger.info("==== CALLING reflect_on_summary... ====")
    print("==== CALLING reflect_on_summary... ====\n\n")
    response = response = completion(
        model=CFG.model,
        messages=[
            {
                "content": reflection_system_prompt.format(
                    research_topic=ctx.deps.research_topic
                ),
                "role": "system",
            },
            {
                "content": f"Identify a knowledge gap and generate a follow-up web search query based on our existing knowledge: {ctx.deps.current_summary}",
                "role": "user",
            },
        ],
        max_tokens=500,
        response_format={"type": "json_object"},
    )
    follow_up_query = json.loads(response.choices[0].message.content)
    ctx.deps.search_query = follow_up_query["follow_up_query"]
    return "continue_or_stop_research"

Ta funkcja asynchroniczna analizuje wygenerowane podsumowanie, identyfikuje luki w wiedzy i generuje kolejne zapytanie do wyszukiwarki internetowej.

In [ ]:
async def finalize_summary(ctx: RunContext[ResearchDeps]) -> str:
    """Finalize the summary"""
    print("==== CALLING finalize_summary... ====")
    all_sources = format_sources(ctx.deps.sources)
    ctx.deps.final_summary = (
        f"## Summary:\n\n{ctx.deps.current_summary}\n\n{all_sources}"
    )
    return f"STOP and return this summary: {ctx.deps.final_summary}"


Ta funkcja asynchroniczna finalizuje podsumowanie, formatuje wszystkie zebrane źródła i zwraca ostateczny wynik.

In [ ]:
async def continue_or_stop_research(ctx: RunContext[ResearchDeps]) -> str:
    """Decide to continue the research or stop based on the follow-up query"""
    print("==== CALLING continue_or_stop_research... ====")
    if ctx.deps.research_loop_count >= CFG.MAX_WEB_SEARCH_LOOPS:
        await finalize_summary(ctx)
        return "finalize_summary"
    else:
        return f"Iterations so far: {ctx.deps.research_loop_count}.\n\ngenerate_search_query"


Ta funkcja asynchroniczna decyduje, czy kontynuować proces badawczy, czy go zakończyć na podstawie liczby wykonanych iteracji wyszukiwania i maksymalnej dozwolonej liczby iteracji.

# Asystent

In [ ]:
default_system_prompt = """You are a researcher. You need to use your tools and provide a research.
You must STOP your research if you have done {max_loop} iterations.
"""
research_agent = Agent(
    model,
    system_prompt=default_system_prompt.format(max_loop=CFG.MAX_WEB_SEARCH_LOOPS),
    deps_type=ResearchDeps,
    tools=[
        Tool(generate_search_query),
        Tool(perform_web_search),
        Tool(summarize_sources),
        Tool(reflect_on_summary),
        Tool(finalize_summary),
        Tool(continue_or_stop_research),
    ],
)

In [ ]:
topic = "how to use SmolLM models?"

research_deps = ResearchDeps(research_topic=topic)
result = research_agent.run_sync(topic, deps=research_deps)

==== CALLING generate_search_query... ====
==== CALLING perform_web_search... ====
==== CALLING summarize_sources... ====
==== CALLING reflect_on_summary... ====


==== CALLING continue_or_stop_research... ====
==== CALLING generate_search_query... ====
==== CALLING perform_web_search... ====
==== CALLING summarize_sources... ====
==== CALLING reflect_on_summary... ====


==== CALLING continue_or_stop_research... ====
==== CALLING finalize_summary... ====
==== CALLING finalize_summary... ====


In [ ]:
Markdown(result.data)

## Summary:

The research provides a detailed guide on using SmolLM for efficient language modeling. It emphasizes the importance of preparing the necessary "ingredients," which include dependencies and configurations, to successfully run the model. Specifically, it details how to execute the SmolLM model using full precision, underscoring the need for careful setup to ensure optimal performance. 

In addition to the foundational setup, the guide elaborates on installing the transformers library as a crucial first step, highlighting the importance of having all libraries and models properly installed before initiating data processing. The article provides specific code snippets for loading the model, such as utilizing `AutoModelForCausalLM.from_pretrained(checkpoint).to(device)` and configuring it for optimized execution with settings like `device_map="auto"` and `torch_dtype=torch.bfloat16`.

Furthermore, it notes that SmolLM has undergone extensive pretraining, consisting of 500,000 steps and utilizing approximately 1 trillion tokens. This extensive training enhances the model's operational efficiency and robustness, making it a valuable tool for both seasoned developers and newcomers interested in language modeling. The guide serves as a comprehensive resource for users looking to effectively utilize this language modeling tool.

### Sources:

1. **How to Use SmolLM: Your Guide to Efficient Language Modeling** - [Fxis](https://fxis.ai/edu/how-to-use-smollm-your-guide-to-efficient-language-modeling/)
2. **How to Use SmolLM: Your Guide to State-of-the-Art Small Language Models** - [Fxis](https://fxis.ai/edu/how-to-use-smollm-your-guide-to-state-of-the-art-small-language-models/)

In [ ]:
Markdown(research_deps.final_summary)

## Summary:

The article provides a detailed guide on using SmolLM for efficient language modeling. It emphasizes the importance of preparing the necessary "ingredients," which include dependencies and configurations, to successfully run the model. Specifically, it details how to execute the SmolLM model using full precision, underscoring the need for careful setup to ensure optimal performance. 

In addition to the foundational setup, the guide elaborates on installing the transformers library as a crucial first step, highlighting the importance of having all libraries and models properly installed before initiating data processing. The article provides specific code snippets for loading the model, such as utilizing `AutoModelForCausalLM.from_pretrained(checkpoint).to(device)` and configuring it for optimized execution with settings like `device_map="auto"` and `torch_dtype=torch.bfloat16`.

Furthermore, it notes that SmolLM has undergone extensive pretraining, consisting of 500,000 steps and utilizing approximately 1 trillion tokens. This extensive training enhances the model's operational efficiency and robustness, making it a valuable tool for both seasoned developers and newcomers interested in language modeling. The guide serves as a comprehensive resource for users looking to effectively utilize this language modeling tool.

Sources:

Source 1:
Title: How to Use SmolLM: Your Guide to Efficient Language Modeling - Fxis
Url: https://fxis.ai/edu/how-to-use-smollm-your-guide-to-efficient-language-modeling/
Content: Step 2: Running the Model Using Full Precision. Imagine preparing to cook a gourmet meal. First, you gather all your high-quality ingredients and tools. Similarly, running the SmolLM model requires setting up your "ingredients" (dependencies and configurations) efficiently. To run the SmolLM model using full precision, follow these steps:

Source 2:
Title: How to Use SmolLM: Your Guide to State-of-the-Art Small Language Models
Url: https://fxis.ai/edu/how-to-use-smollm-your-guide-to-state-of-the-art-small-language-models/
Content: How to Use SmolLM: Your Guide to State-of-the-Art Small Language Models fxis.ai How to Use SmolLM: Your Guide to State-of-the-Art Small Language Models Welcome to our inventive guide on utilizing SmolLM, a series of small yet powerful language models developed for operational efficiency and robustness. Whether you’re a seasoned AI developer or a curious novice, this article will walk you through everything from installation to running your first model. Before you dive into the glorious world of SmolLM, you need to install the transformers library. First, you need all your ingredients (libraries and models) in place, then you can start cooking (processing data). model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device) model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto", torch_dtype=torch.bfloat16) SmolLM went through extensive training comprising 500,000 pretraining steps using a whopping 1 trillion tokens.